In [1]:
import pandas as pd
import numpy as np
from sklearn import datasets
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.svm import SVC

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')
import os

path = r"C:\Users\Frans\koulu\AMK Tietojenkäsittely 2024-2025"

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)

df = pd.read_csv(full_path)
print(df.head())

   Destination Port  Flow Duration  Total Fwd Packets  \
0                22        1266342                 41   
1                22        1319353                 41   
2                22            160                  1   
3                22        1303488                 41   
4             35396             77                  1   

   Total Length of Fwd Packets  Fwd Packet Length Max  Fwd Packet Length Min  \
0                         2664                    456                      0   
1                         2664                    456                      0   
2                            0                      0                      0   
3                         2728                    456                      0   
4                            0                      0                      0   

   Fwd Packet Length Mean  Fwd Packet Length Std  Bwd Packet Length Max  \
0               64.975610             109.864573                    976   
1               64.975610 

In [2]:

df100k = df.sample(n=100000, random_state=42)
 
Valitut10 =[
'Init_Win_bytes_forward',
'Flow IAT Mean',
'Packet Length Std',
'Subflow Fwd Bytes',
'Flow Duration',
'Bwd Packet Length Mean',
'Total Length of Fwd Packets',
'PSH Flag Count',
'Flow Packets/s',
'Destination Port',
'Attack Type'
]

df_valitut = df100k[Valitut10]
print(df_valitut.head())

         Init_Win_bytes_forward  Flow IAT Mean  Packet Length Std  \
2469841                      -1   6.155400e+03          44.403453   
1288855                     274   7.210000e+02         136.541081   
817253                      256   5.626240e+06        1627.461760   
1260920                    8192   1.722652e+06        1026.224887   
1193007                   65535   2.756890e+04         544.924340   

         Subflow Fwd Bytes  Flow Duration  Bwd Packet Length Mean  \
2469841                124          30777              122.000000   
1288855                 53           2884              341.000000   
817253                  56       73141122             1934.500000   
1260920                845      115417660             1216.111111   
1193007               1204         551378              613.500000   

         Total Length of Fwd Packets  PSH Flag Count  Flow Packets/s  \
2469841                          124               0      194.950775   
1288855                   

In [18]:
Valitut10.head()

AttributeError: 'list' object has no attribute 'head'

In [3]:
df_valitut.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 2469841 to 2155658
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Init_Win_bytes_forward       100000 non-null  int64  
 1   Flow IAT Mean                100000 non-null  float64
 2   Packet Length Std            100000 non-null  float64
 3   Subflow Fwd Bytes            100000 non-null  int64  
 4   Flow Duration                100000 non-null  int64  
 5   Bwd Packet Length Mean       100000 non-null  float64
 6   Total Length of Fwd Packets  100000 non-null  int64  
 7   PSH Flag Count               100000 non-null  int64  
 8   Flow Packets/s               100000 non-null  float64
 9   Destination Port             100000 non-null  int64  
 10  Attack Type                  100000 non-null  object 
dtypes: float64(4), int64(6), object(1)
memory usage: 9.2+ MB


In [4]:
X = df_valitut.drop('Attack Type', axis=1)
y = df_valitut['Attack Type']


X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Opetusdatan (Train) koko: {X_train.shape[0]} riviä ({X_train.shape[0]/len(df_valitut)*100}%)")
print(f"Validointidatan (Validation) koko: {X_val.shape[0]} riviä ({X_val.shape[0]/len(df_valitut)*100}%)")
print(f"Testidatan (Test) koko: {X_test.shape[0]} riviä ({X_test.shape[0]/len(df_valitut)*100}%)")

Opetusdatan (Train) koko: 60000 riviä (60.0%)
Validointidatan (Validation) koko: 20000 riviä (20.0%)
Testidatan (Test) koko: 20000 riviä (20.0%)


In [5]:
# Huom: Vaatii imbalanced-learn -kirjaston asennuksen (pip install imbalanced-learn)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# 1. Määritellään putken vaiheet. 
# Tehtävänanto kehotti pohtimaan järjestystä: skaalataanko ensin ja sitten SMOTE, vai toisinpäin?
putki = ImbPipeline([
    ('skaalaus', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('malli', KNeighborsClassifier())
])

# 2. Määritellään testattavat hyperparametrit (GridSearch)
# Kokeillaan esim. eri naapureiden määriä kNN:lle
parametrit = {
    'malli__n_neighbors': [3, 5, 7],
    # Voit myös testata putken vaiheita täällä! Esimerkiksi StandardScaler vs MinMaxScaler.
}

# 3. Luodaan GridSearchCV-olio
grid_search = GridSearchCV(putki, parametrit, cv=3, scoring='accuracy', n_jobs=-1)

# 4. Koulutetaan malli (Tämä vaihe voi kestää hetken!)
# grid_search.fit(X_train, y_train)